In [244]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

In [245]:
purchases = pd.read_csv("amazon-purchases.csv")
survey = pd.read_csv("survey2.csv")

In [246]:
purchases.info(show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1850717 entries, 0 to 1850716
Data columns (total 8 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Order Date                1850717 non-null  object 
 1   Purchase Price Per Unit   1850717 non-null  float64
 2   Quantity                  1850717 non-null  float64
 3   Shipping Address State    1762905 non-null  object 
 4   Title                     1760977 non-null  object 
 5   ASIN/ISBN (Product Code)  1849744 non-null  object 
 6   Category                  1761259 non-null  object 
 7   Survey ResponseID         1850717 non-null  object 
dtypes: float64(2), object(6)
memory usage: 113.0+ MB


In [247]:
purchases_samsung = purchases[
    purchases["Title"].str.contains(r"\bsamsung\b", case=False, regex=True, na=False)
].copy()

purchases_samsung.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 19706 entries, 5 to 1850564
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order Date                19706 non-null  object 
 1   Purchase Price Per Unit   19706 non-null  float64
 2   Quantity                  19706 non-null  float64
 3   Shipping Address State    19136 non-null  object 
 4   Title                     19706 non-null  object 
 5   ASIN/ISBN (Product Code)  19704 non-null  object 
 6   Category                  19699 non-null  object 
 7   Survey ResponseID         19706 non-null  object 
dtypes: float64(2), object(6)
memory usage: 1.4+ MB


In [248]:
purchases_samsung["Shipping Address State"].value_counts(dropna=False)

Shipping Address State
CA     2113
TX     1562
FL     1387
PA     1083
OH      946
NY      875
IL      707
NC      694
GA      602
NaN     570
MI      566
WA      532
VA      530
NJ      520
IN      455
WI      452
KY      433
AZ      384
MA      368
TN      352
CO      337
OR      336
MN      323
AL      291
MD      284
SC      245
MO      234
NV      212
UT      194
OK      190
LA      184
KS      164
AR      159
NE      157
WV      154
NH      153
CT      149
IA      117
MS      111
ME       80
NM       78
DC       67
ID       65
RI       52
HI       48
DE       47
VT       42
MT       28
AK       25
WY       24
SD       19
ND        6
Name: count, dtype: int64

In [249]:
purchases_samsung.isnull().sum()

Order Date                    0
Purchase Price Per Unit       0
Quantity                      0
Shipping Address State      570
Title                         0
ASIN/ISBN (Product Code)      2
Category                      7
Survey ResponseID             0
dtype: int64

Vamos a eliminar los datos nulos porque equivale a 2.9% de los datos para samsung

In [250]:
purchases_samsung=purchases_samsung.dropna()
purchases_samsung.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 19129 entries, 5 to 1850564
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order Date                19129 non-null  object 
 1   Purchase Price Per Unit   19129 non-null  float64
 2   Quantity                  19129 non-null  float64
 3   Shipping Address State    19129 non-null  object 
 4   Title                     19129 non-null  object 
 5   ASIN/ISBN (Product Code)  19129 non-null  object 
 6   Category                  19129 non-null  object 
 7   Survey ResponseID         19129 non-null  object 
dtypes: float64(2), object(6)
memory usage: 1.3+ MB


## Cambio tipo de variables

In [251]:
purchases_samsung["Order Date"]=pd.to_datetime(purchases_samsung["Order Date"], errors="coerce")
purchases_samsung.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 19129 entries, 5 to 1850564
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Order Date                19129 non-null  datetime64[ns]
 1   Purchase Price Per Unit   19129 non-null  float64       
 2   Quantity                  19129 non-null  float64       
 3   Shipping Address State    19129 non-null  object        
 4   Title                     19129 non-null  object        
 5   ASIN/ISBN (Product Code)  19129 non-null  object        
 6   Category                  19129 non-null  object        
 7   Survey ResponseID         19129 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 1.3+ MB


## EDA survey

In [252]:
survey.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5027 entries, 0 to 5026
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Survey ResponseID           5027 non-null   object
 1   Q-demos-age                 5027 non-null   object
 2   Q-demos-hispanic            5027 non-null   object
 3   Q-demos-race                5027 non-null   object
 4   Q-demos-education           5027 non-null   object
 5   Q-demos-income              5027 non-null   object
 6   Q-demos-gender              5027 non-null   object
 7   Q-sexual-orientation        5027 non-null   object
 8   Q-demos-state               5027 non-null   object
 9   Q-amazon-use-howmany        5027 non-null   object
 10  Q-amazon-use-hh-size        5027 non-null   object
 11  Q-amazon-use-how-oft        5027 non-null   object
 12  Q-substance-use-cigarettes  5027 non-null   object
 13  Q-substance-use-marijuana   5027 non-null   obje

Solo el 32.6% de los encuestados quisieron compartir su experiencia de vida en la encuesta por lo que se considera que el 64% de las personas no quisieron compartir esos datos por tal motivo se elimina la columna

In [253]:
survey.drop(columns=["Q-life-changes"], inplace=True)
survey.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5027 entries, 0 to 5026
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Survey ResponseID           5027 non-null   object
 1   Q-demos-age                 5027 non-null   object
 2   Q-demos-hispanic            5027 non-null   object
 3   Q-demos-race                5027 non-null   object
 4   Q-demos-education           5027 non-null   object
 5   Q-demos-income              5027 non-null   object
 6   Q-demos-gender              5027 non-null   object
 7   Q-sexual-orientation        5027 non-null   object
 8   Q-demos-state               5027 non-null   object
 9   Q-amazon-use-howmany        5027 non-null   object
 10  Q-amazon-use-hh-size        5027 non-null   object
 11  Q-amazon-use-how-oft        5027 non-null   object
 12  Q-substance-use-cigarettes  5027 non-null   object
 13  Q-substance-use-marijuana   5027 non-null   obje

## brand samsung + electronic

In [254]:
data_merge=pd.merge(purchases_samsung, survey, left_on="Survey ResponseID", right_on="Survey ResponseID", how="inner")
data_merge.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19129 entries, 0 to 19128
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Order Date                  19129 non-null  datetime64[ns]
 1   Purchase Price Per Unit     19129 non-null  float64       
 2   Quantity                    19129 non-null  float64       
 3   Shipping Address State      19129 non-null  object        
 4   Title                       19129 non-null  object        
 5   ASIN/ISBN (Product Code)    19129 non-null  object        
 6   Category                    19129 non-null  object        
 7   Survey ResponseID           19129 non-null  object        
 8   Q-demos-age                 19129 non-null  object        
 9   Q-demos-hispanic            19129 non-null  object        
 10  Q-demos-race                19129 non-null  object        
 11  Q-demos-education           19129 non-null  object    

In [255]:
pd.set_option("display.max_rows", None)
electronic_category = data_merge["Category"]

## Base de datos para k means

##

In [256]:
# ------------------------------------------------------------
# Renombrar columnas principales
# ------------------------------------------------------------
purchases = purchases.rename(columns={
    "Survey ResponseID": "response_id",
    "Order Date": "order_date",
    "Purchase Price Per Unit": "unit_price",
    "Quantity": "quantity",
    "Shipping Address State": "shipping_state",
    "Title": "title",
    "Category": "category"
})

survey = survey.rename(columns={
    "Survey ResponseID": "response_id"
})

# ------------------------------------------------------------
# Tipos de datos
# ------------------------------------------------------------
purchases["response_id"] = purchases["response_id"].astype(str)
survey["response_id"] = survey["response_id"].astype(str)

purchases["order_date"] = pd.to_datetime(purchases["order_date"], errors="coerce")
purchases["unit_price"] = pd.to_numeric(purchases["unit_price"], errors="coerce")
purchases["quantity"] = pd.to_numeric(purchases["quantity"], errors="coerce")

# Gasto total por línea
purchases["amount"] = purchases["unit_price"] * purchases["quantity"]

# Limpieza básica
purchases["title"] = purchases["title"].fillna("").astype(str)
purchases["category"] = purchases["category"].fillna("").astype(str)
purchases["shipping_state"] = purchases["shipping_state"].fillna("UNKNOWN").astype(str)

In [257]:
# Filtrar primero la ventana T1
T1_START = pd.to_datetime("2018-01-01")
T1_END   = pd.to_datetime("2021-10-31")

purchases_T1 = purchases[
    (purchases["order_date"] >= T1_START) &
    (purchases["order_date"] <= T1_END) &
    (purchases["amount"] > 0)
].copy()

# Crear título en minúsculas
purchases_T1["title_lc"] = purchases_T1["title"].fillna("").astype(str).str.lower()

# Quedarse solo con productos Samsung
focal_T1 = purchases_T1[
    purchases_T1["title_lc"].str.contains(r"\bsamsung\b", regex=True, na=False)
].copy()

focal_T1.shape

(13697, 10)

In [258]:
base_kmeans = (
    focal_T1
    .groupby("response_id")
    .agg(
        recency_days=("order_date", lambda x: (T1_END - x.max()).days),
        frequency=("order_date", "count"),
        monetary=("amount", "sum"),
        avg_ticket=("amount", "mean"),
        total_units=("quantity", "sum"),
        n_categories=("category", "nunique"),
        n_products=("title", "nunique"),
        n_states=("shipping_state", "nunique")
    )
    .reset_index()
)

base_kmeans.head()

,response_id,recency_days,frequency,monetary,avg_ticket,total_units,n_categories,n_products,n_states
0,R_01vNIayewjIIKMF,248,9,270.91,30.101111,9.0,4,5,1
1,R_037XK72IZBJyF69,146,5,57.45,11.490000,5.0,3,5,1
2,R_03aEbghUILs9NxD,23,1,10.99,10.990000,1.0,1,1,1
3,R_06RZP9pS7kONINr,159,2,63.94,31.970000,2.0,2,2,1
4,R_06d9ULxrBmkwSTn,215,7,159.46,22.780000,7.0,6,7,1


In [259]:
survey[survey["response_id"]=="R_01vNIayewjIIKMF"]

,response_id,Q-demos-age,Q-demos-hispanic,Q-demos-race,Q-demos-education,Q-demos-income,Q-demos-gender,Q-sexual-orientation,Q-demos-state,Q-amazon-use-howmany,...,Q-substance-use-cigarettes,Q-substance-use-marijuana,Q-substance-use-alcohol,Q-personal-diabetes,Q-personal-wheelchair,Q-sell-YOUR-data,Q-sell-consumer-data,Q-small-biz-use,Q-census-use,Q-research-society
221,R_01vNIayewjIIKMF,35 - 44 years,Yes,Black or African American,Bachelor's degree,"$25,000 - $49,999",Male,heterosexual (straight),New Jersey,1 (just me!),...,No,No,No,No,No,Yes if I get part of the profit,Yes if consumers get part of the profit,No,No,Yes


In [260]:
train_ids = pd.read_csv("train_ids.csv")["response_id"].astype(str)

base_train = base_kmeans[
    base_kmeans["response_id"].isin(train_ids)
].copy()
print(base_train.shape)
base_train.head()


(1876, 9)


,response_id,recency_days,frequency,monetary,avg_ticket,total_units,n_categories,n_products,n_states
0,R_01vNIayewjIIKMF,248,9,270.91,30.101111,9.0,4,5,1
1,R_037XK72IZBJyF69,146,5,57.45,11.490000,5.0,3,5,1
2,R_03aEbghUILs9NxD,23,1,10.99,10.990000,1.0,1,1,1
4,R_06d9ULxrBmkwSTn,215,7,159.46,22.780000,7.0,6,7,1
5,R_08uYA7fb4unHGkF,48,1,293.00,293.000000,1.0,1,1,1


In [261]:
base_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1876 entries, 0 to 3120
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   response_id   1876 non-null   object 
 1   recency_days  1876 non-null   int64  
 2   frequency     1876 non-null   int64  
 3   monetary      1876 non-null   float64
 4   avg_ticket    1876 non-null   float64
 5   total_units   1876 non-null   float64
 6   n_categories  1876 non-null   int64  
 7   n_products    1876 non-null   int64  
 8   n_states      1876 non-null   int64  
dtypes: float64(3), int64(5), object(1)
memory usage: 146.6+ KB
